In [ ]:
import librosa
import numpy as np
import os
import glob
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import pickle
import warnings
warnings.filterwarnings('ignore')


In [ ]:
emotions = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'suprised'
}

observed_emotions = ['neutral','calm', 'happy', 'sad', 'angry', 'disgust', 'suprised']
print("emotions detected")
print(f"defined emotions: {observed_emotions}")

emotions detected
defined emotions: ['neutral', 'calm', 'happy', 'sad', 'angry', 'disgust', 'suprised']


In [ ]:
def extract_features(file_path):
    audio, sample_rate = librosa.load(file_path, res_type= 'kaiser_fast')
    mfccs = librosa.feature.mfcc(
        y = audio,
        sr = sample_rate,
        n_mfcc=40
    )
    mfcc_mean = np.mean(mfccs.T,axis =0)

    return np.hstack(mfcc_mean)
print("Feature extraction function ready!")

Feature extraction function ready!


In [ ]:
def load_dataset(dataset_path):
    features = []
    labels = []
    files = glob.glob(os.path.join(dataset_path, "Actor_*", "*.wav"))
    print(f"Found {len(files)} audio files")

    for file_path in files:
        filename = os.path.basename(file_path)
        emotion_code = filename.split('-')[2]
        emotion = emotions.get(emotion_code)
        
        if emotion not in observed_emotions:
            continue
        
        feature = extract_features(file_path)
        if feature is None:
            continue
        features.append(feature)
        labels.append(emotion)

    print(f"loaded {len(features)} audio files successfully")
    return np.array(features), np.array(labels)
print("Dataset loaded succesfully")

Dataset loaded succesfully


In [ ]:
dataset_path = "audio_speech_actors_01-24"
print("Loading dataset...")

X,y = load_dataset(dataset_path)
print(f"Total sample : {X.shape[0]}")
print(f"emotions found: {np.unique(y)}")

for emotion in np.unique(y):
    count = np.sum(y == emotion)
    print(f" {emotion} = {count} samples")


Loading dataset...
Found 1440 audio files
loaded 1248 audio files successfully
Total sample : 1248
emotions found: ['angry' 'calm' 'disgust' 'happy' 'neutral' 'sad' 'suprised']
 angry = 192 samples
 calm = 192 samples
 disgust = 192 samples
 happy = 192 samples
 neutral = 96 samples
 sad = 192 samples
 suprised = 192 samples


In [ ]:
scaler = StandardScaler()
X_scaled =scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled,y ,test_size=0.2, random_state=42)

In [ ]:
X_train.shape, X_test.shape

((998, 40), (250, 40))

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes= (300,200,100),
    activation= 'tanh',
    solver= 'adam',
    max_iter= 500,
    random_state= 42,
    verbose= True
)
print("Training model...")
model.fit(X_train,y_train)
print('\n Model trained Successfully')

Training model...
Iteration 1, loss = 1.82500642
Iteration 2, loss = 1.51547902
Iteration 3, loss = 1.38231419
Iteration 4, loss = 1.29176490
Iteration 5, loss = 1.20700449
Iteration 6, loss = 1.15215934
Iteration 7, loss = 1.10444531
Iteration 8, loss = 1.06051607
Iteration 9, loss = 1.02424720
Iteration 10, loss = 0.99627220
Iteration 11, loss = 0.96712286
Iteration 12, loss = 0.93887968
Iteration 13, loss = 0.90821390
Iteration 14, loss = 0.88176820
Iteration 15, loss = 0.84989105
Iteration 16, loss = 0.82564842
Iteration 17, loss = 0.80018862
Iteration 18, loss = 0.77078131
Iteration 19, loss = 0.74481883
Iteration 20, loss = 0.72198820
Iteration 21, loss = 0.69487400
Iteration 22, loss = 0.67186191
Iteration 23, loss = 0.65066835
Iteration 24, loss = 0.62038270
Iteration 25, loss = 0.59784975
Iteration 26, loss = 0.57422542
Iteration 27, loss = 0.55975656
Iteration 28, loss = 0.53167058
Iteration 29, loss = 0.50862709
Iteration 30, loss = 0.48086258
Iteration 31, loss = 0.46470231

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f" Model Accuracy: {accuracy*100:.2f}")
print(f" predicted : {y_pred[:10]}")
print(f"actual : {y_test[:10]}")


 Model Accuracy: 75.60
 predicted : ['neutral' 'disgust' 'happy' 'angry' 'happy' 'suprised' 'suprised' 'angry'
 'disgust' 'calm']
actual : ['neutral' 'suprised' 'happy' 'angry' 'happy' 'suprised' 'suprised'
 'angry' 'disgust' 'happy']


In [ ]:
with open('emotion_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Model saved as emotion_model.plk')
print('Scaler saved as scaler.pkl')

Model saved as emotion_model.plk
Scaler saved as scaler.pkl


In [ ]:
def predict_emotion(file_path):
    with open('emotion_model.pkl', 'rb') as f:
        loaded_model = pickle.load(f)
    
    with open('scaler.pkl', 'rb') as f:
        loaded_scaler = pickle.load(f)

    feature = extract_features(file_path)

    if feature is None:
        print("Error: Could not extract features")

    feature = feature.reshape(1, -1)
    feature_scaled = loaded_scaler.transform(feature)

    prediction = loaded_model.predict(feature_scaled)

    probability = loaded_model.predict_proba(feature_scaled)

    print(f"Audio file: {file_path}")
    print(f"Predicted emotion: {prediction[0].upper()}")
    print(f"confidence Scores:")

    for emotion,prob in zip(loaded_model.classes_, probability[0]):
        bar = '-'*int(prob*20)
        print(f"{emotion:10} {bar} {prob*100:.1f}%")
        print('Prediction function ready')

In [ ]:

test_file = "1001_DFA_HAP_XX.wav"

predict_emotion(test_file)

FileNotFoundError: [Errno 2] No such file or directory: '1001_DFA__XX.wav'

In [ ]:
import glob
import os

# 1. Find all files matching the 06 emotion code
fearful_files = glob.glob("audio_speech_actors_01-24/Actor_*/*-*-02-*-*-*-*.wav")

print(f"Found {len(fearful_files)} Fearful (07) files. Running predictions:\n")

# 2. Extract, Scale, and Predict each one
for file_path in fearful_files:
    features = extract_features(file_path)
    scaled_features = scaler.transform([features])
    prediction = model.predict(scaled_features)[0]
    
    # Print just the filename and what the model thinks it is
    print(f"{os.path.basename(file_path)} ---> Predicted: {prediction.upper()}")

Found 192 Fearful (07) files. Running predictions:

03-01-02-01-01-01-01.wav ---> Predicted: CALM
03-01-02-01-01-02-01.wav ---> Predicted: CALM
03-01-02-01-02-01-01.wav ---> Predicted: CALM
03-01-02-01-02-02-01.wav ---> Predicted: CALM
03-01-02-02-01-01-01.wav ---> Predicted: CALM
03-01-02-02-01-02-01.wav ---> Predicted: CALM
03-01-02-02-02-01-01.wav ---> Predicted: CALM
03-01-02-02-02-02-01.wav ---> Predicted: CALM
03-01-02-01-01-01-02.wav ---> Predicted: CALM
03-01-02-01-01-02-02.wav ---> Predicted: CALM
03-01-02-01-02-01-02.wav ---> Predicted: CALM
03-01-02-01-02-02-02.wav ---> Predicted: CALM
03-01-02-02-01-01-02.wav ---> Predicted: CALM
03-01-02-02-01-02-02.wav ---> Predicted: CALM
03-01-02-02-02-01-02.wav ---> Predicted: CALM
03-01-02-02-02-02-02.wav ---> Predicted: CALM
03-01-02-01-01-01-03.wav ---> Predicted: CALM
03-01-02-01-01-02-03.wav ---> Predicted: CALM
03-01-02-01-02-01-03.wav ---> Predicted: CALM
03-01-02-01-02-02-03.wav ---> Predicted: CALM
03-01-02-02-01-01-03.wav ---